In [23]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

In [24]:
df = pd.read_csv("data/spotify_2015_2025_85k.csv")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


release_col = "release_date"
genre_col = "genre"
stream_col = "stream_count"


df["year"] = pd.to_datetime(df[release_col], errors="coerce").dt.year


df = df[["year", genre_col, stream_col]].copy()
df = df.rename(columns={
    genre_col: "genre",
    stream_col: "stream_count"
})


df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["stream_count"] = pd.to_numeric(df["stream_count"], errors="coerce")
df["genre"] = df["genre"].astype(str).str.strip()

df = df.dropna(subset=["year", "genre", "stream_count"])
df = df[df["genre"] != ""]
df = df[~df["genre"].str.lower().isin(["unknown", "none", "nan"])]
df = df[df["year"].between(2015, 2025)]

In [25]:
# 1. Preparación de datos
all_genres_df = (
    df.groupby(['year', 'genre'], as_index=False)['stream_count']
    .mean()
    .sort_values(['genre', 'year'])
)
all_genres = sorted(all_genres_df['genre'].unique())

base_color = "#b0c4de" 
highlight_color = "#1e90ff" 

#top3
top3_genres = (
    df.groupby("genre", as_index=False)["stream_count"]
    .mean()
    .sort_values("stream_count", ascending=False)
    .head(3)["genre"]
    .tolist()
)


df_top3 = df[df["genre"].isin(top3_genres)].copy()


df_plot = (
    df_top3.groupby(["year", "genre"], as_index=False)["stream_count"]
        .mean()
        .sort_values(["year", "genre"])
)


color_map = {
    top3_genres[0]: "#E29C05",  
    top3_genres[1]: "#B61B82",  
    top3_genres[2]: "#5E6CEC"   
}

fig = go.Figure()

for genre in all_genres:
    genre_data = all_genres_df[all_genres_df['genre'] == genre]
    
    # Línea Principal (Recta)
    fig.add_trace(go.Scatter(
        x=genre_data['year'],
        y=genre_data['stream_count'],
        mode='lines',
        name=genre,
        line=dict(shape='linear', width=3, color=color_map.get(genre, base_color)), 
        hoverinfo='y',
        legendgroup=genre,
        opacity=1
    ))

    # Puntos sutiles
    fig.add_trace(go.Scatter(
        x=genre_data['year'],
        y=genre_data['stream_count'],
        mode='markers',
        marker=dict(color=color_map.get(genre, base_color)),
        showlegend=False,
        hoverinfo='skip',
        legendgroup=genre
    ))

    # Nombre del género al final de la línea
    last_row = genre_data[genre_data['year'] == 2025]
    if not last_row.empty:
        fig.add_trace(go.Scatter(
            x=[2025],
            y=[last_row['stream_count'].iloc[0]],
            mode='text',
            text=[f'    {genre}'],
            textposition='middle right',
            textfont=dict(size=12, color=color_map.get(genre, base_color)),
            showlegend=False,
            hoverinfo='skip',
            legendgroup=genre
        ))

# 2. Configuración del Layout (Título y Espaciado restaurados)
fig.update_layout(
    title=dict(
        text="<b>Evolución del Consumo Promedio de los Géneros Musicales (2015–2025)</b>",
        font=dict(size=24, color="#2c3e50"),
        x=0.02,
        y=0.9, # Posición del título bien arriba
        xanchor='left',
        yanchor='top'
    ),
    xaxis=dict(
        title="Año",
        tickmode="linear",
        dtick=1,
        range=[2014.8, 2027], # Espacio para etiquetas
        showgrid=False,
        linecolor="#d3d3d3"
    ),
    yaxis=dict(
        title="Streams (Promedio Anual)",
        showgrid=True,
        gridcolor="#f0f0f0",
        zeroline=True,
        zerolinecolor="#d3d3d3",
        tickformat="~s",
        range=[0, None] 
    ),
    template="plotly_white",
    width=1400,
    height=750, # Altura para que no haya scroll
    margin=dict(l=80, r=220, t=120, b=100), # t=120 da espacio al título
    plot_bgcolor="white",
    paper_bgcolor="white",
    dragmode=False,
    showlegend=False,
    font=dict(family="Arial, sans-serif", size=13)
)

# 3. Anotaciones restauradas
fig.add_annotation(
    text="<i>Datos basados en el promedio anual de streams por género a nivel global.</i>",
    xref="paper", yref="paper", x=0, y=1.01, 
    showarrow=False, font=dict(size=14, color="#7f8c8d"), align="left"
)

fig.add_annotation(
    text="<b>Simbología:</b> líneas en color representan el Top 3 de géneros con mayor consumo promedio anual.",
    xref="paper", yref="paper",
    x=0, y=0.975,
    showarrow=False,
    font=dict(size=12, color="#7f8c8d"),
    align="left"
)

fig.add_annotation(
    text=(
        "Cada línea representa el consumo promedio anual, no valores acumulados.<br>"
        "<b>Fuente:</b> Spotify Music Analytics Dataset (2015–2025) / Rohit kumar"
    ),
    xref="paper", yref="paper", x=0, y=-0.15,
    showarrow=False, align="left", font=dict(size=11, color="#95a5a6")
)

# 4. Diccionario de mensajes
genre_messages_dict = {
    "Pop": "El <b>Pop</b> domina las listas de streaming globales con ritmos pegadizos.",
    "Rock": "El <b>Rock</b> mantiene una base de oyentes leales con un consumo estable.",
    "Hip Hop": "El <b>Hip Hop</b> ha mostrado un crecimiento masivo desde 2018.",
}
for g in all_genres:
    if g not in genre_messages_dict:
        genre_messages_dict[g] = f"Información detallada para el género <b>{g}</b>."


# 5. Generación de HTML
html_content = f"""
<html>
<head>
  <meta charset='utf-8' />
  <script src='https://cdn.plot.ly/plotly-2.29.0.min.js'></script>
  <style>
    body {{ 
        font-family: Arial, sans-serif; 
        margin: 0; 
        background: #f8f9fa; 
        display: flex; 
        justify-content: center; 
        align-items: center;
        height: 100vh;
        overflow: hidden;
    }}
    #container {{ 
        position: relative; 
        background: white; 
        border-radius: 12px; 
        box-shadow: 0 4px 20px rgba(0,0,0,0.08); 
        width: 1420px;
        height: 820px;
        padding: 10px;
    }}
    #info-box {{
      position: absolute;
      top: 130px;
      right: 40px;
      width: 260px;
      padding: 20px;
      border-radius: 12px;
      background: rgba(255, 255, 255, 0.9);
      backdrop-filter: blur(8px);
      -webkit-backdrop-filter: blur(8px);
      border: 1px solid rgba(0,0,0,0.05);
      box-shadow: 0 8px 24px rgba(0,0,0,0.12);
      display: none;
      z-index: 100;
    }}
    #info-box h3 {{ margin: 0 0 10px; font-size: 18px; color: #1e90ff; border-bottom: 2px solid #eef2f3; padding-bottom: 5px; }}
    #genre-info {{ font-size: 14px; line-height: 1.5; color: #444; }}
  </style>
</head>
<body>
  <div id="container">
    <div id='plot'></div>
    <div id='info-box'>
      <h3 id='genre-name'>Género</h3>
      <div id='genre-info'>Cargando...</div>
    </div>
  </div>

  <script>
    const fig = {json.dumps(fig.to_plotly_json())};
    const genreMessages = {json.dumps(genre_messages_dict)};
    const baseColor = '{base_color}';
    const highlightColor = '{highlight_color}';

    const colorMap = {{
      "Hip-Hop": "#E29C05",
      "R&B": "#B61B82",
      "Rock": "#5E6CEC"
    }};

    function getGenreColor(genre) {{
      return colorMap[genre] || baseColor;
    }}

    const genreSounds = {{
      "Classical": "../Segunda_entrega/Sonidos/CLASSICAL.mp3",
      "Country": "../Segunda_entrega/Sonidos/COUNTRY.mp3",
      "EDM": "../Segunda_entrega/Sonidos/EDM.mp3",
      "Folk": "../Segunda_entrega/Sonidos/FOLK.mp3",
      "Hip-Hop": "../Segunda_entrega/Sonidos/HIP-HOP.mp3",
      "Indie": "../Segunda_entrega/Sonidos/INDIE.mp3",
      "Jazz": "../Segunda_entrega/Sonidos/JAZZ.mp3",
      "Metal": "../Segunda_entrega/Sonidos/METAL.mp3",
      "Pop": "../Segunda_entrega/Sonidos/POP.mp3",
      "R&B": "../Segunda_entrega/Sonidos/R&B.mp3",
      "Reggaeton": "../Segunda_entrega/Sonidos/REGGAETON.mp3",
      "Rock": "../Segunda_entrega/Sonidos/ROCK.mp3"
    }};

    let currentAudio = null;
    let audioEnabled = false;

    const plotDiv = document.getElementById('plot');
    const infoBox = document.getElementById('info-box');
    const genreName = document.getElementById('genre-name');
    const genreInfo = document.getElementById('genre-info');

    let currentHoveredGenre = null;

    document.body.addEventListener('click', function() {{
      if (!audioEnabled) {{
        audioEnabled = true;
        console.log('Audio enabled by click. Ahora pasa el cursor sobre un género para escuchar sonido.');
      }}
    }});

    function playGenreSound(genre) {{
      if (!audioEnabled) {{
        console.warn('Audio bloqueado hasta realizar un clic. Haz clic en el gráfico para habilitar sonido.');
        return;
      }}

      if (currentAudio) {{
        currentAudio.pause();
        currentAudio.currentTime = 0;
      }}

      if (genreSounds[genre]) {{
        currentAudio = new Audio(genreSounds[genre]);
        currentAudio.volume = 0.5;
        currentAudio.play().catch(function(error) {{
          console.warn('Error al reproducir audio:', error);
        }});
      }}
    }}

    function highlightGenre(genre) {{
      if (currentHoveredGenre === genre) return;

      currentHoveredGenre = genre;
      playGenreSound(genre);

      genreName.textContent = genre;
      genreInfo.innerHTML = genreMessages[genre];
      infoBox.style.display = 'block';

      fig.data.forEach(t => {{
        if (t.legendgroup === genre) {{
          t.opacity = 1;
          if (t.mode === 'lines') {{ t.line.color = getGenreColor(t.legendgroup);; t.line.width = 4; }}
          if (t.mode === 'markers') t.marker.color = getGenreColor(t.legendgroup);
          if (t.mode === 'text') t.textfont.color = highlightColor;
        }} else {{
          t.opacity = 0.1;
          if (t.mode === 'lines') {{ t.line.color = getGenreColor(t.legendgroup); t.line.width = 3; }}
          if (t.mode === 'markers') t.marker.color = getGenreColor(t.legendgroup);
          if (t.mode === 'text') t.textfont.color = baseColor;
        }}
      }});

      Plotly.react(plotDiv, fig.data, fig.layout);
    }}

    function resetVisualization() {{
      currentHoveredGenre = null;

      if (currentAudio) {{
        currentAudio.pause();
        currentAudio.currentTime = 0;
      }}

      infoBox.style.display = 'none';

      fig.data.forEach(t => {{
        t.opacity = 1;
        if (t.mode === 'lines') {{ t.line.color = getGenreColor(t.legendgroup); t.line.width = 3; }}
        if (t.mode === 'markers') t.marker.color = getGenreColor(t.legendgroup);
        if (t.mode === 'text') t.textfont.color = getGenreColor(t.legendgroup);
      }});

      Plotly.react(plotDiv, fig.data, fig.layout);
    }}

    Plotly.newPlot(plotDiv, fig.data, fig.layout, {{ responsive: true, displayModeBar: false }});

    plotDiv.on('plotly_hover', function(data) {{
      if (data.points.length > 0) {{
        const hoveredGenre = fig.data[data.points[0].curveNumber].legendgroup;
        highlightGenre(hoveredGenre);
      }}
    }});

    plotDiv.on('plotly_unhover', function() {{
      resetVisualization();
    }});
  </script>
</body>
</html>
"""

# Guardado final
output_path = os.path.join("..", "docs", "index.html")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)